# 03 - ELO Prediction Model

Fit a regression model that predicts a player's ELO from their Average Centipawn Loss (ACPL).

Uses the 200k sampled Lichess games with real Stockfish evals to calibrate:
1. Compute ACPL per player per game from the moves CSV (real Stockfish centipawns)
2. Join with known ELO from the games CSV
3. Fit `ELO = a - b × ln(ACPL)` using nonlinear regression
4. Also fit a multivariate model (ACPL + blunder rate + captures + checks)
5. PCA player profile clustering
6. Export coefficients to `elo_model.json` for the React frontend

## Setup

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("scikit-learn")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import json
import os

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

print('Setup done.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Learning-document/Data Visualization/Project 2/data'
FRONTEND_DIR = os.path.join(DATA_DIR, 'frontend')
os.makedirs(FRONTEND_DIR, exist_ok=True)

games = pd.read_csv(os.path.join(DATA_DIR, 'lichess_sampled_games.csv'))
moves = pd.read_csv(os.path.join(DATA_DIR, 'lichess_sampled_moves.csv'))

print(f'Games: {len(games):,}')
print(f'Moves (eval): {len(moves):,}')
print(f'\nMoves columns: {list(moves.columns)}')
print(f'Games columns: {list(games.columns)}')

## 1. Compute ACPL per player per game

From the moves CSV:
- `eval_change_for_player`: centipawn change from the player's perspective (negative = blunder)
- ACPL = average of max(0, -eval_change) across all moves in the game
- Join with the player's known ELO from the games CSV

In [ ]:
# Cap individual eval changes at ±1000cp to reduce noise from mate scores
MAX_EVAL_CHANGE = 1000
MIN_MOVES = 10  # skip very short games

moves['eval_change_clipped'] = moves['eval_change_for_player'].clip(-MAX_EVAL_CHANGE, MAX_EVAL_CHANGE)
moves['loss_cp'] = moves['eval_change_clipped'].clip(lower=0) * -1  # positive = centipawns lost
moves['loss_cp'] = moves['eval_change_clipped'].apply(lambda x: max(0, -x))

# Group by (game_idx, is_white) = one player's performance in one game
player_stats = moves.groupby(['game_idx', 'is_white']).agg(
    acpl=('loss_cp', 'mean'),
    n_moves=('loss_cp', 'count'),
    total_loss=('loss_cp', 'sum'),
    blunder_count=('eval_change_clipped', lambda x: (x < -200).sum()),
    capture_count=('is_capture', 'sum'),
    check_count=('is_check', 'sum'),
).reset_index()

player_stats['blunder_rate'] = player_stats['blunder_count'] / player_stats['n_moves']
player_stats['capture_pct'] = player_stats['capture_count'] / player_stats['n_moves']
player_stats['check_pct'] = player_stats['check_count'] / player_stats['n_moves']

# Filter short games
player_stats = player_stats[player_stats['n_moves'] >= MIN_MOVES]

# Join with ELO
def get_elo(row):
    game = games.iloc[int(row['game_idx'])]
    return game['white_rating'] if row['is_white'] else game['black_rating']

player_stats['elo'] = player_stats.apply(get_elo, axis=1)

# Filter unreasonable values
mask = (player_stats['acpl'] > 0) & (player_stats['acpl'] < 1000) & \
       (player_stats['elo'] > 400) & (player_stats['elo'] < 3200)
df = player_stats[mask].copy()

print(f'Player-game records: {len(df):,}')
print(f'\nACPL stats:')
print(df['acpl'].describe().to_string())
print(f'\nELO stats:')
print(df['elo'].describe().to_string())

## 2. Visualize ACPL vs ELO

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Scatter (sampled for visibility)
sample = df.sample(min(10000, len(df)), random_state=42)
ax1.scatter(sample['acpl'], sample['elo'], alpha=0.08, s=4, c='steelblue')
ax1.set_xlabel('ACPL (centipawns)')
ax1.set_ylabel('ELO')
ax1.set_title('ACPL vs ELO (per player-game)')
ax1.set_xlim(0, 400)
ax1.set_ylim(400, 3000)

# Binned averages
elo_bins = np.arange(400, 3001, 100)
df['elo_bin'] = pd.cut(df['elo'], elo_bins)
binned = df.groupby('elo_bin', observed=True).agg(
    acpl_mean=('acpl', 'mean'),
    acpl_std=('acpl', 'std'),
    count=('acpl', 'count'),
).dropna()
binned = binned[binned['count'] >= 20]
midpoints = [(b.left + b.right) / 2 for b in binned.index]

ax2.errorbar(midpoints, binned['acpl_mean'],
             yerr=binned['acpl_std'] / np.sqrt(binned['count']),
             fmt='o-', capsize=3, markersize=4, linewidth=1.5, c='steelblue')
ax2.set_xlabel('ELO')
ax2.set_ylabel('Average ACPL (centipawns)')
ax2.set_title('Average ACPL by ELO Bracket')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

print('Clear inverse relationship: higher ELO = lower ACPL (fewer mistakes)')

## 3. Fit Simple Regression: ELO = a - b × ln(ACPL)

In [ ]:
def elo_from_acpl(acpl, a, b):
    """ELO = a - b * ln(ACPL), clamped to avoid log(0)."""
    return a - b * np.log(np.maximum(acpl, 1))

# Fit on binned data for stability (avoids overweighting dense regions)
binned_fit = df.groupby('elo_bin', observed=True).agg(
    acpl_mean=('acpl', 'mean'),
    elo_mean=('elo', 'mean'),
    count=('elo', 'count'),
).dropna()
binned_fit = binned_fit[binned_fit['count'] >= 20]

popt, pcov = curve_fit(
    elo_from_acpl,
    binned_fit['acpl_mean'].values,
    binned_fit['elo_mean'].values,
    p0=[3000, 600],  # initial guess from literature
)
a_simple, b_simple = popt
a_ci = 1.96 * np.sqrt(pcov[0, 0])
b_ci = 1.96 * np.sqrt(pcov[1, 1])

print(f'Simple model: ELO = {a_simple:.1f} - {b_simple:.1f} × ln(ACPL)')
print(f'  a = {a_simple:.1f} ± {a_ci:.1f}')
print(f'  b = {b_simple:.1f} ± {b_ci:.1f}')

# R² on raw data
elo_pred = elo_from_acpl(df['acpl'].values, *popt)
ss_res = np.sum((df['elo'].values - elo_pred) ** 2)
ss_tot = np.sum((df['elo'].values - df['elo'].mean()) ** 2)
r2_simple = 1 - ss_res / ss_tot
print(f'  R² = {r2_simple:.3f}')

# Expected ACPL for each ELO bracket
print(f'\nExpected ACPL by ELO:')
for elo in [800, 1200, 1600, 2000, 2400, 2800]:
    expected_acpl = np.exp((a_simple - elo) / b_simple)
    print(f'  ELO {elo}: ACPL ≈ {expected_acpl:.0f} cp')

In [ ]:
# Visualize the fit
fig, ax = plt.subplots(figsize=(10, 6))

acpl_range = np.linspace(5, 400, 200)
elo_pred_curve = elo_from_acpl(acpl_range, *popt)
ax.plot(acpl_range, elo_pred_curve, 'r-', linewidth=2.5,
        label=f'ELO = {a_simple:.0f} - {b_simple:.0f} × ln(ACPL)')

# Binned data points
sizes = binned_fit['count'] / binned_fit['count'].max() * 150
ax.scatter(binned_fit['acpl_mean'], binned_fit['elo_mean'],
           s=sizes, alpha=0.6, c='steelblue', edgecolors='navy', zorder=5)

# ELO bracket bands
for elo, label in [(800, '0-1000'), (1200, '1000-1400'), (1600, '1400-1800'),
                   (2000, '1800-2200'), (2400, '2200-2600')]:
    ax.axhline(y=elo, color='gray', linestyle=':', alpha=0.3)

ax.set_xlabel('ACPL (centipawns)', fontsize=12)
ax.set_ylabel('ELO', fontsize=12)
ax.set_title(f'ACPL → ELO Regression  (R² = {r2_simple:.3f}, n = {len(df):,})', fontsize=13)
ax.legend(fontsize=11, loc='upper right')
ax.set_xlim(0, 400)
ax.set_ylim(400, 3000)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Fit Multivariate Model

Add blunder rate, capture %, and check % as additional features.
Uses log-transformed ACPL + standardized features.

In [ ]:
feature_names = ['ln_acpl', 'blunder_rate', 'capture_pct', 'check_pct']

X = pd.DataFrame({
    'ln_acpl': np.log(df['acpl'].clip(lower=1)),
    'blunder_rate': df['blunder_rate'],
    'capture_pct': df['capture_pct'],
    'check_pct': df['check_pct'],
}).values

y = df['elo'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

reg = LinearRegression()
reg.fit(X_scaled, y)
r2_multi = reg.score(X_scaled, y)

print(f'Multivariate regression (R² = {r2_multi:.3f}):')
print(f'  Intercept: {reg.intercept_:.1f}')
for name, coef in zip(feature_names, reg.coef_):
    print(f'  {name:>15}: {coef:>8.1f}')

# Compare with simple model
print(f'\nSimple model R²:     {r2_simple:.3f}')
print(f'Multivariate R²:     {r2_multi:.3f}')
print(f'Improvement:         +{r2_multi - r2_simple:.3f}')

## 5. LDA: Player Profile Clustering

Supervised dimensionality reduction (Linear Discriminant Analysis) on per-player-game behavioral features.
Unlike PCA, LDA uses era labels to find the directions that maximally separate eras.
Features: ACPL, blunder rate, capture %, check %, number of moves.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Join period from games CSV onto player_stats
period_map = games.set_index(games.index)['period']
df['period'] = df['game_idx'].map(period_map)

# Select features for LDA
lda_features = ['acpl', 'blunder_rate', 'capture_pct', 'check_pct', 'n_moves']
lda_df = df.dropna(subset=lda_features).copy()

# Standardize
lda_scaler = StandardScaler()
X_lda_scaled = lda_scaler.fit_transform(lda_df[lda_features].values)
y_era = lda_df['period'].values

# Run LDA (supervised: uses era labels to maximize between-class separation)
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda_transformed = lda.fit_transform(X_lda_scaled, y_era)
lda_df['ld1'] = X_lda_transformed[:, 0]
lda_df['ld2'] = X_lda_transformed[:, 1]

print(f'LDA explained variance ratio:')
print(f'  LD1: {lda.explained_variance_ratio_[0]:.3f}')
print(f'  LD2: {lda.explained_variance_ratio_[1]:.3f}')
print(f'  Total: {sum(lda.explained_variance_ratio_):.3f}')
print(f'\nClassification accuracy: {lda.score(X_lda_scaled, y_era):.3f}')
print(f'\nLoadings (feature → LD1, LD2):')
for i, feat in enumerate(lda_features):
    print(f'  {feat:>15}: {lda.scalings_[i][0]:+.3f}, {lda.scalings_[i][1]:+.3f}')

# Also train Random Forest for feature importances
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_lda_scaled, y_era)
print(f'\nRandom Forest accuracy: {rf.score(X_lda_scaled, y_era):.3f}')
print(f'Feature importances:')
for feat, imp in sorted(zip(lda_features, rf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {feat:>15}: {imp:.3f}')

# Stratified sample ~5000 (equal per period, oversample small groups)
N_SAMPLE = 1250  # 1250 per period × 4 = 5000
sampled = lda_df.groupby('period', group_keys=False).apply(
    lambda x: x.sample(n=N_SAMPLE, replace=len(x) < N_SAMPLE, random_state=42)
)
print(f'\nSampled {len(sampled)} rows from {len(lda_df)} total')
print(sampled['period'].value_counts().to_string())

# Export CSV
export_cols = ['ld1', 'ld2', 'period', 'elo'] + lda_features
lda_csv_path = os.path.join(FRONTEND_DIR, 'player_pca.csv')
sampled[export_cols].to_csv(lda_csv_path, index=False)
print(f'\nExported to: {lda_csv_path}')
print(f'File size: {os.path.getsize(lda_csv_path)} bytes')

## 6. Export Model to JSON

Export both the simple formula and the multivariate model.
The frontend uses the simple formula with a scaling factor to convert
in-browser eval units to approximate centipawns.

In [ ]:
model = {
    # --- Simple formula: ELO = a - b * ln(ACPL_cp) ---
    'type': 'acpl_regression',
    'formula': f'ELO = {a_simple:.1f} - {b_simple:.1f} * ln(ACPL)',
    'a': round(a_simple, 2),
    'b': round(b_simple, 2),

    # --- Scaling: browser eval uses piece-value units (1 pawn = 1 unit)
    # --- Real eval uses centipawns (1 pawn = 100 cp)
    # --- eval_to_cp = factor to multiply browser ACPL by to get approximate centipawns
    'eval_to_cp': 100,

    # --- Multivariate model (for more accurate prediction if features available) ---
    'multi_feature_names': feature_names,
    'multi_scaler_mean': scaler.mean_.tolist(),
    'multi_scaler_scale': scaler.scale_.tolist(),
    'multi_coefs': reg.coef_.tolist(),
    'multi_intercept': round(reg.intercept_, 2),

    # --- Gaussian kernel for ELO bracket probabilities ---
    'classes': ['0-1000', '1000-1400', '1400-1800', '1800-2200', '2200-2600', '2600+'],
    'centers': [500, 1200, 1600, 2000, 2400, 2800],
    'sigma': 300,

    # --- Metadata ---
    'n_samples': len(df),
    'r_squared_simple': round(r2_simple, 3),
    'r_squared_multi': round(r2_multi, 3),
}

model_path = os.path.join(FRONTEND_DIR, 'elo_model.json')
with open(model_path, 'w') as f:
    json.dump(model, f, indent=2)

print(f'Exported to: {model_path}')
print(f'File size: {os.path.getsize(model_path)} bytes')
print(f'\nModel summary:')
print(json.dumps(model, indent=2))

In [ ]:
# Validation table: what ELO does the model predict for each bracket's expected ACPL?
print('Validation: model predictions vs expected ELO')
print(f'{"ELO":>6} | {"Actual ACPL":>12} | {"Predicted ELO":>14} | {"Error":>8}')
print('-' * 55)

for elo_target in [800, 1200, 1600, 2000, 2400, 2800]:
    # Find the average ACPL for games near this ELO
    nearby = df[(df['elo'] >= elo_target - 200) & (df['elo'] < elo_target + 200)]
    if len(nearby) == 0:
        continue
    actual_acpl = nearby['acpl'].mean()
    predicted_elo = elo_from_acpl(actual_acpl, *popt)
    error = predicted_elo - elo_target
    print(f'{elo_target:>6} | {actual_acpl:>12.1f} | {predicted_elo:>14.1f} | {error:>+8.1f}')

## 7. Copy to frontend

After running this notebook, copy the exported `elo_model.json` to the React frontend.

In [ ]:
print('Exported files in frontend dir:')
for f in sorted(os.listdir(FRONTEND_DIR)):
    if 'elo' in f.lower():
        path = os.path.join(FRONTEND_DIR, f)
        size = os.path.getsize(path)
        print(f'  {f} ({size} bytes)')

print(f'\nCopy to frontend/public/data/ for deployment.')
print(f'Source: {FRONTEND_DIR}/elo_model.json')
print(f'Target: frontend/public/data/elo_model.json')